# 🚍 교통약자가 많은 지역에 저상버스가 더 많이 다니고 있는가?\n### 한겨레 × (재단법인) 숲과나눔 주최 「AI와 함께하는 교통문제 해결을 위한 데이터 분석 공모전」\n\n- **연구 질문**: 노선버스 대폐차 시 저상버스 의무화 도입 이후, 저상버스는 실제로 장애인·고령인구 밀집지역에 우선 배정되었는가?\n- **분석 데이터**: 전국 17개 광역 시·도 및 경기도 31개 기초지자체(6,431개 버스 노선) 전수 데이터\n- **핵심 결과 요약**: \n  1. 전국 단위 단순 상관분석(n=17, df=15)에서 저상버스 도입률과 교통약자 비율 간 음(-)의 상관관계 관찰 ( = -0.432, t = -1.85, p = 0.084$), 등록장애인 비율 기준  = -0.571 (p = 0.017 < 0.05)$, 재정자립도 기준  = +0.619 (p = 0.008 < 0.05)$\n  2. 경기도 노선 전수 단순 상관분석(n=31, df=29)에서 저상버스 노선 비율과 교통약자 비율 간 유의미한 음의 상관성 확증 ( = -0.430, t = -2.56, p = 0.016 < 0.05$)\n  3. 경기도 다중회귀분석(OLS: n=31, 잔차 df=28) 결과: 모형 F(2, 28) = 3.23, p ≈ 0.055. 교통약자 비율 계수  = -1.14 (SE = 0.80, t = -1.42, p = 0.168) 음(-)의 계수가 추정됨\n  4. 화면 표기는 소수점 첫째 자리 반올림 표기값이며, 모든 통계 검정은 반올림 전 원자료로 정밀 계산됨

## 1. 분석 환경 설정 및 데이터 전처리 마스터셋 로드
- 출처: 국토교통부 『2023년 교통약자 이동편의 실태조사 보고서』(TMACS), 통계청 KOSIS, 경기데이터드림

In [ ]:
import pandas as pd
import numpy as np
import math

# 1. 전국 17개 시·도 데이터셋
df_sido = pd.read_csv('../data/sido_master_complete.csv')
# 2. 경기도 31개 시·군 노선 집계 데이터셋
df_gg = pd.read_csv('../data/gyeonggi_master_analysis.csv').dropna(subset=['low_floor_route_ratio', 'vulnerable_rate', 'fiscal_rate'])

print(f'전국 시·도 레코드 수: {len(df_sido)}개 | 경기도 시·군 레코드 수: {len(df_gg)}개')
df_sido[['region', 'total_rate', 'city_bus_rate', 'vulnerable_rate', 'disabled_rate', 'fiscal_rate']].head()

## 2. 전국 17개 시·도 상관분석 및 검정통계량 (t값, exact p-value 계산)
- $t = r \times \sqrt{\frac{n-2}{1 - r^2}}$ 수식을 이용한 엄밀한 가설 검정

In [ ]:
# t분포 누적분포함수 및 p-value 계산 함수
def t_cdf(t, df):
    def pdf(u):
        c = math.gamma((df+1)/2.0) / (math.sqrt(df * math.pi) * math.gamma(df/2.0))
        return c * (1.0 + u*u/df)**(-(df+1)/2.0)
    steps = 10000
    a, b = -20.0, t
    if b < a: return 0.0
    h = (b - a) / steps
    s = pdf(a) + pdf(b)
    for i in range(1, steps):
        u = a + i * h
        s += 4*pdf(u) if i % 2 == 1 else 2*pdf(u)
    return s * h / 3.0

def two_tailed_p(t_val, df):
    return 2.0 * t_cdf(-abs(t_val), df)

n_sido = len(df_sido)
df_sido_deg = n_sido - 2

for col, name in [('vulnerable_rate', '교통약자 통합비율'), ('disabled_rate', '등록장애인 비율'), ('fiscal_rate', '재정자립도')]:
    r = df_sido['total_rate'].corr(df_sido[col])
    t = r * np.sqrt(df_sido_deg / (1 - r**2))
    p = two_tailed_p(t, df_sido_deg)
    print(f'[{name}] 상관계수 r = {r:.4f}, t = {t:.4f}, p = {p:.4f}')

## 3. 경기도 31개 시·군 OLS 다중회귀분석 (표준오차, t값, p값 산출 코드)
- 종속변수: 시군별 저상버스 운행 노선 비율 (%)
- 독립변수: 교통약자 비율 (%), 통제변수: 재정자립도 (%)

In [ ]:
n_gg = len(df_gg)
X = np.column_stack([np.ones(n_gg), df_gg['vulnerable_rate'].values, df_gg['fiscal_rate'].values])
y = df_gg['low_floor_route_ratio'].values
k = X.shape[1] # 3
df_resid = n_gg - k # 28

beta = np.linalg.inv(X.T @ X) @ X.T @ y
y_pred = X @ beta
resid = y - y_pred
s2 = np.sum(resid**2) / df_resid
cov_b = s2 * np.linalg.inv(X.T @ X)
se_b = np.sqrt(np.diag(cov_b))
t_b = beta / se_b
p_b = [two_tailed_p(t, df_resid) for t in t_b]

r2 = 1 - np.sum(resid**2) / np.sum((y - np.mean(y))**2)
adj_r2 = 1 - (1 - r2) * (n_gg - 1) / df_resid

summary_df = pd.DataFrame({
    '변수명': ['Intercept', '교통약자비율(X1)', '재정자립도(X2)'],
    '회귀계수(Beta)': np.round(beta, 4),
    '표준오차(SE)': np.round(se_b, 4),
    't-statistic': np.round(t_b, 4),
    'p-value': np.round(p_b, 4)
})

print('=== 경기도 31개 시·군 다중회귀분석(OLS) 결과 ===')
print(summary_df.to_string(index=False))
print(f'R-squared: {r2:.4f}, Adjusted R-squared: {adj_r2:.4f}')

## 4. 데이터 기반 저상버스 우선순위 산정 모델(안) (LBEI 시뮬레이션 예시)\n- **목적**: 단순 지자체별 균등 신청에 의존하지 않고, 취약계층 밀도와 시설 접근성, 현재 공급 결핍도를 결합한 탐색적 지수 산출 방안(제안안)\n- **정규화 및 산식**: 교통약자 비율(15~40% 기준 Min-Max 스케일링), 시설접근도(가중치 0.3), 공급결핍도(1 - 저상노선비율, 가중치 0.3)\n- **한계**: 본 산식은 정책 대안 모색을 위한 시뮬레이션 예시이며, 실제 도입 시에는 AHP 등을 통한 가중치 정밀 검증이 선행되어야 함

In [ ]:
def calculate_lbei(vulnerable_ratio, welfare_score, current_supply_ratio):
    # 0~1 스케일링
    norm_vul = (vulnerable_ratio - 15) / (40 - 15)
    norm_sup = current_supply_ratio / 100.0
    lbei = (0.40 * norm_vul) + (0.30 * welfare_score) + (0.30 * (1.0 - norm_sup))
    return np.clip(lbei * 100, 0, 100)

df_gg['LBEI_Score'] = calculate_lbei(df_gg['vulnerable_rate'], 0.5, df_gg['low_floor_route_ratio'])
print('=== LBEI 점수 기준 우선 배정 대상 기초지자체 상위 10 ===')
print(df_gg[['관할시군', 'vulnerable_rate', 'low_floor_route_ratio', 'fiscal_rate', 'LBEI_Score']].sort_values(by='LBEI_Score', ascending=False).head(10).to_string(index=False))